# Bulletin blueprint — W19 banking demo

Test E2E pipeline: data card → script → audio → render. Sample data inline (`SAMPLE_DATA`, `RAW_FROM_CLAUDE`) — thay bằng data thật + paste-từ-Claude sau.

**Yêu cầu trước khi chạy:** `.env` có `VBEE_TOKEN` + `VBEE_APPID`. Cell 6 cần Node.js + Remotion components (chưa có → Cell 6 sẽ fail intentional).


In [1]:
# Cell 1: Imports + config
import sys
sys.path.insert(0, '..')

import json
import datetime
from pathlib import Path

from lib import schema, validator, vbee, audio, cache, render

PROJECT_ROOT = Path('..').resolve()
TEMPLATE = "bulletin"

# Inputs - đổi mỗi video mới
VIDEO_ID = "2026-05-12_W19_banking"
WEEK = 19
TOPIC = "Banking tuần W19 — biến động mạnh"

VOICE = "hn_male_phuthang_stor80dt_48k-fhg"  # HN - Anh Khôi, storytelling (more prosody)
SPEED = 0.95  # slower for natural pacing

# Setup paths
OUTPUT_DIR = PROJECT_ROOT / "outputs" / VIDEO_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = PROJECT_ROOT / "remotion" / "public" / "runs" / VIDEO_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"OUTPUT_DIR   = {OUTPUT_DIR}")
print(f"RUN_DIR      = {RUN_DIR}")


PROJECT_ROOT = D:\twan_projects\video-report
OUTPUT_DIR   = D:\twan_projects\video-report\outputs\2026-05-12_W19_banking
RUN_DIR      = D:\twan_projects\video-report\remotion\public\runs\2026-05-12_W19_banking


d:\twan_projects\video-report\python\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


## Cell 2 — Build data card

TODO Phase 2: thay `SAMPLE_DATA` bằng `data_card.build(week=WEEK)` đọc từ JSON file thật.

In [2]:
# Cell 2: Build data card from SAMPLE_DATA (inline placeholder)
SAMPLE_DATA = {
    "week": WEEK,
    "ending_date": "2026-05-09",
    "sector": "Ngân hàng",
    "top_gainers": [
        {"ticker": "TCB", "close": 52300, "change_pct": 8.5},
        {"ticker": "VCB", "close": 105400, "change_pct": 6.2},
        {"ticker": "MBB", "close": 28900, "change_pct": 5.7},
    ],
    "sector_metrics": {
        "roe_current": 18.2,
        "roe_previous": 16.5,
    },
}

def build_data_card(data: dict) -> str:
    lines = [
        f"# Data card — Tuần W{data['week']} (kết thúc {data['ending_date']})",
        f"",
        f"## Ngành: {data['sector']}",
        f"",
        f"## Top tăng giá tuần",
    ]
    for item in data['top_gainers']:
        lines.append(f"- {item['ticker']}: {item['close']:,} đồng ({item['change_pct']:+.1f}%)")
    lines.append("")
    lines.append("## Chỉ số ngành")
    sm = data['sector_metrics']
    delta = sm['roe_current'] - sm['roe_previous']
    lines.append(f"- ROE bình quân hiện tại: {sm['roe_current']:.1f}%")
    lines.append(f"- ROE bình quân quý trước: {sm['roe_previous']:.1f}%")
    lines.append(f"- Delta: {delta:+.1f} điểm phần trăm")
    return "\n".join(lines)

card_md = build_data_card(SAMPLE_DATA)
(OUTPUT_DIR / "data_card.md").write_text(card_md, encoding="utf-8")
print(card_md)
print("---")
print(f"Saved: {OUTPUT_DIR / 'data_card.md'}")


# Data card — Tuần W19 (kết thúc 2026-05-09)

## Ngành: Ngân hàng

## Top tăng giá tuần
- TCB: 52,300 đồng (+8.5%)
- VCB: 105,400 đồng (+6.2%)
- MBB: 28,900 đồng (+5.7%)

## Chỉ số ngành
- ROE bình quân hiện tại: 18.2%
- ROE bình quân quý trước: 16.5%
- Delta: +1.7 điểm phần trăm
---
Saved: D:\twan_projects\video-report\outputs\2026-05-12_W19_banking\data_card.md


## Cell 3 — PASTE script JSON từ Claude

Flow thật: copy `data_card.md` ở Cell 2 → Claude.ai Project `Video — Bulletin` → Claude trả JSON → paste vào đây.

Now: dùng `RAW_FROM_CLAUDE` đã viết tay (đúng format Claude sẽ trả) để demo.

In [3]:
# Cell 3: PASTE script JSON từ Claude (raw string giữa """ """)
RAW_FROM_CLAUDE = """
{
  "video_id": "2026-05-12_W19_banking",
  "template": "bulletin",
  "meta": {
    "title": "Bản tin Banking W19/2026",
    "voice": "hn_male_phuthang_news65dt_44k-fhg",
    "speed": 1.05,
    "fps": 30,
    "width": 1080,
    "height": 1920,
    "created_at": "2026-05-12T20:00:00+07:00"
  },
  "scenes": [
    {
      "id": "s01",
      "type": "headline",
      "narration_text": "Tuần mười chín, cổ phiếu ngân hàng dẫn đầu thị trường với mức tăng bình quân năm phẩy bảy phần trăm. Techcombank, Vietcombank và MB tiếp tục là động lực chính.",
      "data": {
        "headline": "Banking dẫn đầu W19",
        "category": "Thị trường",
        "issue_label": "Tuần 19/2026"
      }
    },
    {
      "id": "s02",
      "type": "quick_rank",
      "narration_text": "Top ba ngân hàng tăng mạnh. Techcombank dẫn đầu với mức tăng tám phẩy năm phần trăm, đóng cửa năm mươi hai nghìn ba trăm đồng. Vietcombank đứng thứ hai với sáu phẩy hai phần trăm.",
      "data": {
        "title": "Top tăng giá ngành ngân hàng",
        "items": [
          {"rank": 1, "label": "TCB", "value": 52300, "change_pct": 8.5},
          {"rank": 2, "label": "VCB", "value": 105400, "change_pct": 6.2},
          {"rank": 3, "label": "MBB", "value": 28900, "change_pct": 5.7}
        ]
      }
    },
    {
      "id": "s03",
      "type": "kpi",
      "narration_text": "Tỷ suất sinh lời trên vốn chủ sở hữu của ngành ngân hàng đạt mười tám phẩy hai phần trăm, tăng một phẩy bảy điểm phần trăm so với quý trước.",
      "data": {
        "metric": "ROE bình quân ngành",
        "current_value": 18.2,
        "previous_value": 16.5,
        "unit": "%",
        "delta": 1.7
      }
    },
    {
      "id": "s04",
      "type": "outro",
      "narration_text": "Theo dõi VBSE để cập nhật phân tích tuần tới. Cảm ơn các bạn đã lắng nghe.",
      "data": {
        "cta": "Theo dõi VBSE",
        "handle": "@vbse"
      }
    }
  ]
}
"""

print(f"RAW length: {len(RAW_FROM_CLAUDE)} chars")


RAW length: 1920 chars


## Cell 4 — Parse + validate

Layer 1 (Pydantic) + Layer 3 (content sanity). Nếu fail, error message paste lại Claude để sửa.

In [4]:
# Cell 4: Parse + validate, loop nếu fail
try:
    raw_dict = validator.parse_claude_output(RAW_FROM_CLAUDE)
    script = schema.Script.model_validate(raw_dict)
    errors = validator.check_content_sanity(script)

    if errors:
        msg = validator.format_errors_for_claude(errors)
        print(msg)
        raise ValueError("Validate fail - paste msg trên cho Claude fix")

    (OUTPUT_DIR / "script_final.json").write_text(
        script.model_dump_json(indent=2), encoding="utf-8"
    )
    print(f"OK: {len(script.scenes)} scenes")
    for s in script.scenes:
        print(f"  [{s.id}] {s.type}: {s.narration_text[:60]}...")
except Exception as e:
    print(f"LỖI: {e}")
    raise


OK: 4 scenes
  [s01] headline: Tuần mười chín, cổ phiếu ngân hàng dẫn đầu thị trường với mứ...
  [s02] quick_rank: Top ba ngân hàng tăng mạnh. Techcombank dẫn đầu với mức tăng...
  [s03] kpi: Tỷ suất sinh lời trên vốn chủ sở hữu của ngành ngân hàng đạt...
  [s04] outro: Theo dõi VBSE để cập nhật phân tích tuần tới. Cảm ơn các bạn...


## Cell 5 — Enrich audio (Vbee + cache + postprocess)

Lần đầu chạy: ~5-15s/scene qua Vbee API. Lần sau: cache hit qua hash narration text.

In [5]:
# Cell 5: Enrich audio
client = vbee.VbeeClient()
audio_dir = RUN_DIR / "audio"
audio_dir.mkdir(parents=True, exist_ok=True)

for scene in script.scenes:
    audio_p = cache.audio_path(scene, audio_dir, voice=VOICE, speed=SPEED)
    if not audio_p.exists():
        print(f"[{scene.id}] Synthesizing ({len(scene.narration_text)} chars)...")
        client.synthesize(scene.narration_text, audio_p, voice=VOICE, speed=SPEED)
        audio.trim_silence(audio_p)
        audio.normalize_peak(audio_p)
    else:
        print(f"[{scene.id}] Cache hit: {audio_p.name}")
    scene.audio_path = f"audio/{audio_p.name}"
    scene.duration = audio.get_duration(audio_p)

# Save enriched script for Remotion to read
(RUN_DIR / "script.json").write_text(
    script.model_dump_json(indent=2), encoding="utf-8"
)

total = sum(s.duration for s in script.scenes)
print(f"\nTotal: {total:.1f}s, {len(script.scenes)} scenes ready")
print(f"Saved: {RUN_DIR / 'script.json'}")


[s01] Cache hit: s01_9398dbcfca67.mp3
[s02] Cache hit: s02_464bfab68938.mp3
[s03] Cache hit: s03_a38be49f7286.mp3
[s04] Cache hit: s04_a0b8f3c04096.mp3

Total: 32.1s, 4 scenes ready
Saved: D:\twan_projects\video-report\remotion\public\runs\2026-05-12_W19_banking\script.json


## Cell 6 — Render Remotion

**BLOCKED** đến khi: (1) cài Node.js, (2) `cd remotion && npm install`, (3) viết Remotion bulletin Composition + 4 scene components.

In [6]:
# Cell 6: Render Remotion (sẽ fail nếu Node/Remotion chưa setup)
output_path = render.render_video(
    template=TEMPLATE,
    video_id=VIDEO_ID,
    project_root=PROJECT_ROOT,
)
print(f"Render xong: {output_path}")

Render xong: D:\twan_projects\video-report\outputs\2026-05-12_W19_banking\render.mp4


## Cell 7 — Preview inline

In [7]:
# Cell 7: Preview
from IPython.display import Video
Video(str(output_path), width=360)


## Cell 8 — Save manifest

In [8]:
# Cell 8: Manifest
manifest = {
    "video_id": VIDEO_ID,
    "template": TEMPLATE,
    "topic": TOPIC,
    "week": WEEK,
    "voice": VOICE,
    "speed": SPEED,
    "scenes_count": len(script.scenes),
    "total_duration_sec": round(sum(s.duration for s in script.scenes), 3),
    "created_at": datetime.datetime.now().astimezone().isoformat(),
}
(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(manifest, indent=2, ensure_ascii=False))


{
  "video_id": "2026-05-12_W19_banking",
  "template": "bulletin",
  "topic": "Banking tuần W19 — biến động mạnh",
  "week": 19,
  "voice": "hn_male_phuthang_stor80dt_48k-fhg",
  "speed": 0.95,
  "scenes_count": 4,
  "total_duration_sec": 32.078,
  "created_at": "2026-05-12T11:18:28.128774+07:00"
}
